In [4]:
# Importacao de arquivos
import sys
import os
sys.path.append(os.path.abspath('..')) 
%load_ext autoreload
%autoreload 2
from src import *

# Importacao de bibliotecas
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, random_split
import time
import itertools
# Configurações gerais
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'xpu' if hasattr(torch, 'xpu') and torch.xpu.is_available() else 'cpu')
BATCH_SIZE = 16

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Parte 2

In [ ]:
def training(model, dataloader, device, peso_fronteira, peso_interior, lr=1e-5, num_epochs=10, use_dice_loss: bool = True):

    pesos = torch.tensor([1.0, peso_interior, peso_fronteira], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=pesos)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    num_batches = len(dataloader)

    print(f"Treinando em: {device}")
    print(f"Batches por época: {num_batches}")

    history = {'loss': [], 'iou': [], 'dice': []}
    for epoch in range(num_epochs):
        model.train()

        accumulated_loss = torch.tensor(0.0, device=device)
        total_intersection = torch.tensor(0.0, device=device)
        total_union = torch.tensor(0.0, device=device)

        for images, masks, _ in dataloader:
            images = images.to(device, non_blocking=True, )
            masks = masks.to(device, dtype = torch.long, non_blocking=True)
            optimizer.zero_grad()

            prediction = model(images)
            loss = criterion(prediction, masks)

    
            if use_dice_loss:
                probs = torch.softmax(prediction, dim=1)
                loss_dice_interior = peso_interior * dice_loss(probs, masks, cass_idx=1)
                loss_dice_fronteira = peso_fronteira * dice_loss(probs, masks, class_idx=2)
                loss+= (loss_dice_fronteira+loss_dice_interior)
            
            loss.backward()
            optimizer.step()

            with torch.no_grad(): # validacao
                pred_class = torch.argmax(prediction, dim=1)
                
                # Para acompanhamento durante o treino, calculamos o IoU/Dice 
                # focado apenas na classe "interior" (índice 1)
                interior_pred = (pred_class == 1).float()
                interior_mask = (masks == 1).float()
                
                intersection = (interior_pred * interior_mask).sum()
                union = interior_pred.sum() + interior_mask.sum() - intersection
                
                accumulated_loss += loss.detach()
                total_intersection += intersection
                total_union += union

        epoch_loss = accumulated_loss.item() / num_batches
        ti = total_intersection.item()
        tu = total_union.item()

        epoch_iou = ti / (tu + 1e-6)
        epoch_dice = (2.0 * ti) / (tu + ti + 1e-6)

        history['loss'].append(epoch_loss)
        history['iou'].append(epoch_iou)
        history['dice'].append(epoch_dice)

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | IoU: {epoch_iou:.4f} | Dice: {epoch_dice:.4f}")

    return history


In [ ]:
# Consigurações
model = UNetTernary()
model.to(DEVICE)
part = 2

# Dataset
path_train = Path('../data/stage1_train')
path_test = Path('../data/stage1_test')
full_dataset = DSB2018Dataset(root_dir=path_train, img_size=128, part=part, cache_in_memory=True)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
test_dataset = DSB2018Dataset(root_dir=path_test, img_size=128, part=part, cache_in_memory=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)




hiperparametros_trilha_a = {
        'espessura_fronteira': [1, 2, 3], # em pixels
        'peso_fronteira': [5.0, 10.0, 20.0], # multiplicador para a classe minoritária
        'peso_interior': [1.0, 1.5]
    }
chaves = list(hiperparametros_trilha_a.keys())
combinacoes = list(itertools.product(*hiperparametros_trilha_a.values()))

melhor_map = -1.0
melhor_config = None

print(f"Iniciando busca em grid com {len(combinacoes)} combinações...\n")

for i, valores in enumerate(combinacoes):
    config_atual = dict(zip(chaves, valores))
    
    print(f"[{i+1}/{len(combinacoes)}] Testando: {config_atual}")
    
    map_validacao = training(**config_atual)



KeyboardInterrupt: 

In [ ]:
real_mAP, real_count_errors, real_densities, real_samples = evaluate2(model, val_loader, DEVICE)

# Plotando métricas e amostras
plot_metrics(real_mAP, real_count_errors, real_densities)
plot_samples(real_samples, part=part)